# 11 — Final Evaluation & Model Freeze

## Objective
Retrain the frozen detector on train+validation and evaluate exactly once on the locked future test period.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Refit on train+validation

In [4]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
import joblib
fraud,_=load_data(); 

train,val,test,bounds=chronological_split(fraud)
trainval=pd.concat([train,val],ignore_index=True)

b=HistoryFeatureBuilder().fit(trainval); 
F=b.transform(trainval); 
T=b.transform(test); 
cols=feature_columns(F)


pre=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())])
A=pre.fit_transform(F[cols]); C=pre.transform(T[cols])
model=IsolationForest(n_estimators=500,random_state=42,n_jobs=-1,contamination="auto").fit(A)
test_score=-model.decision_function(C)


# Threshold is selected from validation only, then applied once to test.
vb=HistoryFeatureBuilder().fit(train)
V=vb.transform(val); Tr=vb.transform(train)
vpre=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())])
VA=vpre.fit_transform(Tr[cols]); VV=vpre.transform(V[cols])
vmodel=IsolationForest(n_estimators=500,random_state=42,n_jobs=-1,contamination="auto").fit(VA)
val_score=-vmodel.decision_function(VV)
threshold=float(np.percentile(val_score,95))
metrics=ranking_metrics(test["class"],test_score); tm=threshold_metrics(test["class"],test_score,threshold)
display(pd.DataFrame([{**metrics,**tm}]))
save_json({"ranking":metrics,"threshold_metrics":tm,"threshold":threshold,"features":cols,"bounds":bounds},REP/"final_test_evaluation.json")


In [6]:
metrics=ranking_metrics(test['class'],test_score); 
val_b=HistoryFeatureBuilder().fit(train); 
V=val_b.transform(val); 
VB=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]).fit(train_b:=val_b.transform(train)[cols]); 
val_model=IsolationForest(n_estimators=500,random_state=42,n_jobs=-1).fit(VB.transform(val_b.transform(train)[cols])); 
val_score=-val_model.decision_function(VB.transform(V[cols])); threshold=float(np.percentile(val_score,95)); 
tm=threshold_metrics(test['class'],test_score,threshold); 
display(pd.DataFrame([{**metrics,**tm}])); 
save_json({"ranking":metrics,"threshold_metrics":tm,"threshold":threshold,"features":cols,"bounds":bounds},
         REP/"final_test_evaluation.json")

,pr_auc,roc_auc,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_500,recall_at_500,threshold,accuracy,precision,recall,f1,balanced_accuracy,tn,fp,fn,tp
0,0.047557,0.502724,0.06,0.002901,0.05,0.004836,0.038,0.018375,0.02245,0.907663,0.049362,0.056093,0.052512,0.502229,20516,1117,976,58


## 3. Freeze deployment artifact

In [ ]:
joblib.dump({"preprocessor":pre,"model":model,
             "features":cols,"history_builder":b},
             ART/"final_model.joblib"); 
display(test.assign(anomaly_score=test_score).sort_values
        ("anomaly_score",ascending=False)
        [["user_id","device_id","ip_address","purchase_value","purchase_time","class","anomaly_score"]].head(30)); 
px.histogram(test.assign(anomaly_score=test_score),
             x="anomaly_score",
             color="class",
             nbins=80,barmode="overlay",
             title="Locked test score distribution").show()

,user_id,device_id,ip_address,purchase_value,purchase_time,class,anomaly_score
148432,192192,EQYVNEGOFLAWK,8.598994e+08,96,2015-09-22 06:38:58+00:00,1,0.200503
128584,118353,CQTUVBYIWWWBC,1.486003e+09,36,2015-10-24 22:10:31+00:00,0,0.176369
150619,67350,ECSJFQBUBSKDH,1.696626e+09,60,2015-11-27 00:49:35+00:00,0,0.158958
36982,379751,CMXZWTEQYRLRU,2.356257e+09,106,2015-10-11 21:21:00+00:00,0,0.142471
66864,315063,UIKTAQMPWLBLI,3.992859e+09,101,2015-10-25 14:24:58+00:00,0,0.134828
118531,81141,MPORHXNBCECHH,2.862993e+09,107,2015-10-25 19:12:35+00:00,0,0.127898
88152,93542,WRTKFZMFEGZLY,4.211873e+07,46,2015-11-30 16:47:25+00:00,0,0.125648
41178,28577,UTZFYONAEQLZS,8.023110e+08,62,2015-09-23 18:42:11+00:00,0,0.123990
99617,162861,SQOBOZSFRHEHX,2.960785e+09,107,2015-10-03 20:36:08+00:00,0,0.123790
116282,341429,HRGSVTFUURPOY,1.265890e+09,92,2015-10-17 23:47:52+00:00,0,0.123317
